# G9 - where is the cliff, and what sets its location?

Six explanations have failed the same way: SMOOTH where the outcome is discontinuous. This tests two candidates that have a threshold BY CONSTRUCTION.

**H1** the cliff sits at the SOURCE encoder ambient dimension (img_small is 768-d; the cliff is at 512->768). **H2** it sits at the head target dimension. Varying source {768, 1536, 2048} against target {bge 1024, sbert 768} separates them completely.

Requires G0-exact to have passed. One SVD, sliced across seven widths.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G9 — where is the cliff, and what sets its location?
# Requires G0-exact to have passed. Runs on the ORIGINAL hub protocol.
#
# THE OBSERVATION THAT MOTIVATES THIS. Six explanations for the width
# cliff have been tested and all six falsified, and every one failed the
# same way: it was SMOOTH where the outcome is discontinuous. So the
# thing to look for is a quantity that has a threshold BY CONSTRUCTION.
#
# There is one sitting in plain sight. The head is trained on img_small,
# which is 768-dimensional. The cliff is at 512 -> 768. Below it, every
# hub width is a projection DOWN from a 768-d source; at 768 the entry map
# becomes square, and the whitening divides by singular values from the
# very tail of a source that has no more directions to give. That is a
# crossing point, not a curve.
#
# TWO HYPOTHESES, and they are separable:
#   H1  the cliff sits at the SOURCE encoder's ambient dimension
#   H2  the cliff sits at the HEAD TARGET's dimension
# Both predict 768 for the published configuration (img_small source,
# and bge is 1024 so H2 predicts 1024 - already slightly off). Vary both
# axes and they come apart completely:
#
#            source   img_small 768   img_base 1536   img_large 2048
#   H1 says  cliff at        768            1536            2048
#   H2 says  cliff at   the target dim, unchanged by the source
#
#   target   bge 1024    sbert 768
#   H1 says  unchanged by the target
#   H2 says       1024          768
#
# If the cliff tracks the source dimension, H1 is supported and the
# mechanism is the entry map running out of source directions. If it sits
# at a fixed width regardless of both, BOTH are falsified and it becomes
# the seventh and eighth entries in the ledger. Either outcome is worth
# the run; a null here is as reportable as a hit.
#
# WHY SBERT IS WORTH INCLUDING BEYOND H2. The report notes SBERT
# outperforms bge as a target (101.0% vs 98.1%) and that bge was kept for
# comparability with every ceiling in the report. This sweep is not bound
# by that comparability, so it can carry both - and if the cliff is
# invariant to the target, that is itself evidence the head is not the
# thing that breaks.
#
# PRE-REGISTERED, before running: the cliff is the largest NEGATIVE step
# in the zero-shot transfer curve, and it counts as located only if that
# step is at least 3x the magnitude of the largest step elsewhere in the
# same curve. A curve with no step meeting that bar is recorded as NO
# CLIFF rather than as a cliff in a new place.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
ALPHA, N_EVAL, SEED = 1e-2, 1000, 0
WIDTHS = [256, 384, 512, 640, 768, 896, 1024]
CLIFF_RATIO = 3.0

In [ ]:
# ---------- the four hub spaces, exactly as G0-exact ----------
SPACES = {}
for size in ("small", "base", "large"):
    SPACES[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N_PAIRS = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N_PAIRS] for k, v in SPACES.items()}
SPACES["txt_bge"] = np.load(
    str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N_PAIRS]

# the second head target. Same deterministic prefix as every other cache
# (E1.3 wrote it over the ids the DINOv2 runs used), so the [:N_PAIRS]
# slice is the same alignment assumption the original made - not a new one.
TARGETS = {"bge": SPACES["txt_bge"]}
_sb = DATA_DIR / "e13_txt_sbert.npz"
if _sb.exists():
    TARGETS["sbert"] = np.load(str(_sb))["txt"].astype(np.float64)[:N_PAIRS]
else:
    print("NOTE: e13_txt_sbert.npz missing - the target axis collapses to")
    print("bge alone, which cannot separate H1 from H2. Find it if you can.")

SOURCES = ["img_small", "img_base", "img_large"]
print(f"N_PAIRS {N_PAIRS}")
for k, v in SPACES.items():
    print(f"  {k:10s} {v.shape[1]:5d}-d")
for k, v in TARGETS.items():
    print(f"  target {k:5s} {v.shape[1]:5d}-d")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N_PAIRS)
te, tr = perm[:N_EVAL], perm[N_EVAL:]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    S = l2n(P) @ l2n(G).T
    return float((S.argmax(1) == np.arange(len(P))).mean())

In [ ]:
# ---------- ONE SVD, sliced per width ----------
# BASIS at width d is V[:d].T / (sigma[:d]/sqrt(n)), so a single
# decomposition serves every width in the sweep. Recomputing it per width
# would be seven times the cost and identical arithmetic.
_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
print(f"\nconcat {_ref.shape}, one SVD reused across {len(WIDTHS)} widths")
assert max(WIDTHS) <= len(_sv), "hub width exceeds the concat rank"


def hub_at(d):
    B = _VT[:d].T / (_sv[:d] / np.sqrt(len(_ref)))
    return (_ref - _mu) @ B


def sweep(src, tgt_name):
    T = TARGETS[tgt_name]
    out = []
    for d in WIDTHS:
        H = hub_at(d)
        to_hub = {k: ridge(SPACES[k][tr], H) for k in SPACES if k.startswith("img_")}
        head = ridge(SPACES[src][tr] @ to_hub[src], T[tr])
        gal = l2n(T[te])
        # zero-shot: mean over the image encoders the head never saw
        pcts = []
        for enc in SOURCES:
            if enc == src:
                continue
            zero = r1((SPACES[enc][te] @ to_hub[enc]) @ head, gal)
            nat = r1(SPACES[enc][te] @ ridge(SPACES[enc][tr], T[tr]), gal)
            pcts.append(zero / max(nat, 1e-9))
        out.append(float(np.mean(pcts)))
    return np.array(out)


def find_cliff(curve):
    """Largest negative step, and whether it clears the pre-registered bar."""
    steps = np.diff(curve)
    i = int(np.argmin(steps))
    if steps[i] >= 0:
        return None, 0.0, steps
    others = np.abs(np.delete(steps, i))
    ratio = abs(steps[i]) / (others.max() + 1e-12)
    return (WIDTHS[i + 1] if ratio >= CLIFF_RATIO else None), ratio, steps


print("\n" + "=" * 86)
print(f"{'source':12s}{'src dim':>8s}{'target':>8s}{'tgt dim':>8s}   "
      + "".join(f"{w:>7d}" for w in WIDTHS) + f"{'cliff':>8s}{'ratio':>7s}")
print("=" * 86)

results = {}
for tgt in TARGETS:
    for src in SOURCES:
        c = sweep(src, tgt)
        at, ratio, _ = find_cliff(c)
        results[(src, tgt)] = (c, at, ratio)
        print(f"{src:12s}{SPACES[src].shape[1]:>8d}{tgt:>8s}"
              f"{TARGETS[tgt].shape[1]:>8d}   "
              + "".join(f"{v:>7.3f}" for v in c)
              + (f"{at:>8d}" if at else f"{'none':>8s}") + f"{ratio:>7.1f}")

In [ ]:
# ---------- read it ----------
print("\n" + "=" * 86)
found = {k: v[1] for k, v in results.items() if v[1]}
if not found:
    print("NO CLIFF LOCATED anywhere in the sweep at the pre-registered bar.")
    print("That is a substantive result, not a failed run: it would mean the")
    print("published cliff does not survive a finer width grid, and the")
    print("0.489 -> 0.312 step should be re-examined at 640 and 896 before")
    print("anything further is built on it.")
else:
    by_src = {src: {at for (s_, _), at in found.items() if s_ == src}
              for src in SOURCES}
    by_tgt = {tgt: {at for (_, t_), at in found.items() if t_ == tgt}
              for tgt in TARGETS}
    for src in SOURCES:
        d_ = SPACES[src].shape[1]
        seen = by_src[src] or "none"
        inrange = d_ <= max(WIDTHS)
        note = "" if inrange else f"  (predicted {d_}, OUTSIDE the swept grid)"
        print(f"  source {src:12s} ({d_:4d}-d): cliff at {seen}{note}")
    for tgt in TARGETS:
        print(f"  target {tgt:12s} ({TARGETS[tgt].shape[1]:4d}-d): "
              f"cliff at {by_tgt[tgt] or 'none'}")

    # H1: for every source whose dimension falls INSIDE the swept grid, the
    # cliff sits at that dimension. Sources whose dimension is beyond the
    # grid are UNINFORMATIVE, not counterevidence - an earlier version of
    # this block counted their absence as a fixed-location result and
    # printed the opposite conclusion.
    testable = [s_ for s_ in SOURCES if SPACES[s_].shape[1] <= max(WIDTHS)]
    untestable = [s_ for s_ in SOURCES if s_ not in testable]
    at_own_dim, contradict = [], []
    for s_ in testable:
        d_ = SPACES[s_].shape[1]
        near = {a for a in by_src[s_] if abs(a - d_) <= 128}
        (at_own_dim if near else contradict).append(s_)
    target_invariant = all(len(v) <= 1 for v in by_tgt.values()) and \
        len({next(iter(v)) for v in by_tgt.values() if v}) <= 1

    print()
    if untestable:
        print(f"  NOTE: {', '.join(untestable)} have ambient dimensions beyond")
        print(f"  the swept maximum of {max(WIDTHS)}, so this run cannot see")
        print("  their cliffs either way. They are uninformative here, NOT")
        print("  evidence against a source-dimension account.")
    if at_own_dim and not contradict:
        print()
        print("H1 SUPPORTED. Every source whose dimension falls inside the")
        print("grid cliffs AT its own ambient dimension, and the location does")
        print("not move with the head target - so H2 is falsified.")
        print()
        print("MECHANISM. A d-dimensional source's ridge entry map spans at")
        print("most d hub directions, so the head fitted on its coordinates")
        print("has never seen the rest. Past width d, other encoders populate")
        print("directions the head cannot read. That is a rank wall - a")
        print("threshold by construction - which is exactly the shape the six")
        print("previously falsified accounts all lacked.")
        print()
        print("THIS IS NOT YET AN EXPLANATION. It fits one source. Extend the")
        print("grid past 1536 and 2048 and the account PREDICTS that img_base")
        print("and img_large cliff at their own dimensions. State that before")
        print("measuring. If they do not, this joins the ledger.")
    elif contradict:
        print("H1 CONTRADICTED on: " + ", ".join(contradict))
        print("A testable source failed to cliff at its own dimension, so the")
        print("rank-wall account is out. Record it as a further ledger entry.")
    else:
        print("NO TESTABLE SOURCE. Every source dimension lies outside the")
        print("swept grid, so neither hypothesis was actually examined.")
        print("Widen WIDTHS and re-run before drawing anything.")
    if not target_invariant:
        print("\n  CAUTION: the cliff location differs by TARGET, so H2 is")
        print("  not cleanly falsified. Report both axes.")

print("\nScope: one hub protocol, one image corpus, three source encoders,")
print("two targets. The sweep varies width on a 128-point grid, so a cliff")
print("is located to within one grid step, not exactly.")